# Building an Interactive NBA Analytics Dashboard with Plotly

This tutorial walks through building a multi-chart interactive dashboard using NBA player data from 1996–2023. We use Plotly for visualization and deploy it as a static HTML page on GitHub Pages.

**Live Dashboard**: [snehithn.com/dashboard](https://snehithn.com/dashboard.html)

---
## 1. Visualization Technique

Four charts + one dropdown for season selection. Selecting a season updates everything at once.

| Chart | What it shows | Why this chart type |
|-------|--------------|---------------------|
| **Line** | League avg PPG, RPG, APG per season | Best for showing trends over time |
| **Bar** (horizontal) | Top 10 scorers for selected season | Easy to compare ranked items; horizontal fits long names |
| **Scatter** | Points vs Assists per player (size = games, color = shooting efficiency) | Shows relationships between two stats + encodes extra dimensions |
| **Histogram** | Age distribution for selected season | Shows the shape/spread of a single variable |

Each chart operates at a different level — the line chart shows league-wide trends, the bar chart ranks individuals, the scatter plot compares players across multiple dimensions, and the histogram shows the overall population.

**Interactivity**: One season dropdown controls all four charts. Hover tooltips on every chart provide detail on demand. The scatter plot uses a red-to-green color scale for True Shooting % so you can immediately see who's efficient.

---
## 2. Visualization Library

### Why Plotly?

I used Plotly — Python (`plotly`) for prototyping here and Plotly.js for the deployed dashboard.

- **Created by**: Plotly Technologies Inc. (Montreal, 2013) GitHub](https://github.com/plotly/plotly.py)
- Declarative (describe what you want and Plotly handles the rendering)
- **Jupyter integration**: Charts render inline with full interactivity (hover, zoom, pan) which is what we're using.

### Installation

```bash
pip install plotly pandas
```

For the deployed HTML dashboard, just a CDN link:
```html
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
```

### Comparison

| | Plotly | Bokeh | D3.js |
|---|---|---|---|
| Built-in interactivity | Yes | Yes | Manual |
| Python + JS | Yes | Python only | JS only |
| Standalone HTML | Yes | Yes | Manual |
| Learning curve | Low-Med | Medium | High |
| Jupyter support | Native | Native | Limited |

Pros
- I've used it before 
- I can use the same library in the html and ipynb file
- Standalone HTML with no backend for GitHub Pages

### Cons

- **Bundle size**: ~3.5MB — heavy 
- **Cross-chart linking**: Standalone Plotly.js needs custom JS for charts to talk to each other (Dash handles this but requires a server)
- **Performance**: Can lag with tens of thousands of points

---
## 3. Demonstration

### 3.1 Dataset

[NBA Players Dataset](https://www.kaggle.com/datasets/justinas/nba-players-data) from Kaggle (Justinas Cirtautas). Per-season stats for every NBA roster player from 1996-97 through 2022-23.

- **Size**: 12,844 rows x 21 columns
- **Key columns**: `player_name`, `team_abbreviation`, `age`, `season`, `gp`, `pts`, `reb`, `ast`, `ts_pct`, `net_rating`
- **Other**: `player_height`, `player_weight`, `college`, `country`, `draft_year/round/number`, `usg_pct`

In [1]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

In [2]:
df = pd.read_csv('data/nba_players.csv')
print(f"Shape: {df.shape}")
print(f"Seasons: {df['season'].nunique()} ({df['season'].min()} to {df['season'].max()})")
df.head()

Shape: (12844, 21)
Seasons: 27 (1996-97 to 2022-23)


,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,...,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,42,...,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,34,...,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,12,...,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,7,...,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,22,...,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


### 3.2 Data Cleaning

Pretty clean out of the box:
1. Removed the unnamed index column (done during preprocessing)
2. No missing values in key columns
3. Data types are correct

In [3]:
key_cols = ['player_name', 'team_abbreviation', 'age', 'pts', 'reb', 'ast', 'season', 'gp', 'ts_pct']
print("Missing values:")
print(df[key_cols].isnull().sum())
print(f"\nData types:\n{df[key_cols].dtypes}")

Missing values:
player_name          0
team_abbreviation    0
age                  0
pts                  0
reb                  0
ast                  0
season               0
gp                   0
ts_pct               0
dtype: int64

Data types:
player_name              str
team_abbreviation        str
age                  float64
pts                  float64
reb                  float64
ast                  float64
season                   str
gp                     int64
ts_pct               float64
dtype: object


In [4]:
df[['pts', 'reb', 'ast', 'age', 'gp']].describe().round(1)

,pts,reb,ast,age,gp
count,12844.0,12844.0,12844.0,12844.0,12844.0
mean,8.2,3.6,1.8,27.0,51.2
std,6.0,2.5,1.8,4.3,25.1
min,0.0,0.0,0.0,18.0,1.0
25%,3.6,1.8,0.6,24.0,31.0
50%,6.7,3.0,1.2,26.0,57.0
75%,11.5,4.7,2.4,30.0,73.0
max,36.1,16.3,11.7,44.0,85.0


No missing values, types look good. Ready to visualize.

### 3.3 Building Each Chart

#### Chart 1: Line Chart — League Averages

In [5]:
season_avgs = df.groupby('season')[['pts', 'reb', 'ast']].mean().reset_index()

fig_line = go.Figure()
fig_line.add_trace(go.Scatter(x=season_avgs['season'], y=season_avgs['pts'], name='PPG', line=dict(color='#FB9633', width=3)))
fig_line.add_trace(go.Scatter(x=season_avgs['season'], y=season_avgs['reb'], name='RPG', line=dict(color='#3b82f6', width=2)))
fig_line.add_trace(go.Scatter(x=season_avgs['season'], y=season_avgs['ast'], name='APG', line=dict(color='#22c55e', width=2)))

fig_line.update_layout(
    title='League Average Stats Over Time',
    xaxis=dict(title='Season', tickangle=-45, dtick=2, type='category'),
    yaxis=dict(title='Per Game Average'),
    template='plotly_dark', height=400
)
fig_line.show()

Key takeaways:
- Scoring dipped mid-2000s, climbing since ~2014 (three-point revolution, faster pace)
- Rebounds stable around 3.5-4 RPG
- Assists trending up — more ball movement in modern offenses

#### Chart 2: Bar Chart — Top 10 Scorers

In [6]:
season = '2022-23'
top10 = df[(df['season'] == season) & (df['gp'] >= 20)].nlargest(10, 'pts')

fig_bar = go.Figure(go.Bar(
    y=top10['player_name'].iloc[::-1], x=top10['pts'].iloc[::-1],
    orientation='h', marker_color='#FB9633',
    text=top10['team_abbreviation'].iloc[::-1],
    textposition='inside', textfont=dict(color='white')
))
fig_bar.update_layout(title=f'Top 10 Scorers ({season})', xaxis_title='Points Per Game',
                      template='plotly_dark', height=400)
fig_bar.show()

Embiid led 2022-23 with 33.1 PPG, followed by Luka and Dame.

#### Chart 3: Scatter Plot — Points vs. Assists

In [7]:
scatter_df = df[(df['season'] == season) & (df['gp'] >= 10)].copy()

fig_scatter = px.scatter(
    scatter_df, x='ast', y='pts', size='gp', color='ts_pct',
    hover_name='player_name',
    hover_data={'team_abbreviation': True, 'gp': True, 'ts_pct': ':.1%'},
    color_continuous_scale=[[0, '#ef4444'], [0.5, '#FB9633'], [1, '#22c55e']],
    labels={'ast': 'Assists Per Game', 'pts': 'Points Per Game', 'ts_pct': 'TS%', 'gp': 'Games Played'},
    title=f'Points vs Assists ({season}) — Size = GP, Color = TS%'
)
fig_scatter.update_layout(template='plotly_dark', height=500)
fig_scatter.show()

Player archetypes:
- **Top-right**: Elite scorers who also distribute (Luka, Harden)
- **Top-left**: Pure scorers, few assists
- **Green dots**: Efficient players (high TS%) — often big men finishing at the rim

#### Chart 4: Histogram — Age Distribution

In [8]:
age_df = df[df['season'] == season]
avg_age = age_df['age'].mean()

fig_hist = go.Figure(go.Histogram(
    x=age_df['age'], xbins=dict(start=18, end=45, size=1),
    marker=dict(color='rgba(251,150,51,0.6)', line=dict(color='#FB9633', width=1))
))
fig_hist.add_vline(x=avg_age, line_dash='dash', line_color='#3b82f6',
                   annotation_text=f'Avg: {avg_age:.1f}', annotation_position='top')
fig_hist.update_layout(title=f'Player Age Distribution ({season})',
                       xaxis_title='Age', yaxis_title='Number of Players',
                       template='plotly_dark', height=400)
fig_hist.show()

Right-skewed as expected. Average age in 2022-23 was ~26.2.

### 3.4 Combined Dashboard View

Using `make_subplots` to preview all four together:

In [9]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('League Averages Over Time', f'Top 10 Scorers ({season})',
                    f'Points vs Assists ({season})', f'Age Distribution ({season})'),
    horizontal_spacing=0.12, vertical_spacing=0.12
)

for col_name, color, name in [('pts', '#FB9633', 'PPG'), ('reb', '#3b82f6', 'RPG'), ('ast', '#22c55e', 'APG')]:
    fig.add_trace(go.Scatter(x=season_avgs['season'], y=season_avgs[col_name], name=name,
                             line=dict(color=color, width=2)), row=1, col=1)

fig.add_trace(go.Bar(y=top10['player_name'].iloc[::-1], x=top10['pts'].iloc[::-1],
                     orientation='h', marker_color='#FB9633', name='PPG',
                     text=top10['team_abbreviation'].iloc[::-1], textposition='inside',
                     showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(x=scatter_df['ast'], y=scatter_df['pts'], mode='markers', name='Players',
                         marker=dict(size=scatter_df['gp']/8, color=scatter_df['ts_pct'],
                                     colorscale=[[0,'#ef4444'],[0.5,'#FB9633'],[1,'#22c55e']],
                                     opacity=0.7),
                         text=scatter_df['player_name'], showlegend=False), row=2, col=1)

fig.add_trace(go.Histogram(x=age_df['age'], xbins=dict(start=18, end=45, size=1),
                           marker=dict(color='rgba(251,150,51,0.6)', line=dict(color='#FB9633', width=1)),
                           name='Age', showlegend=False), row=2, col=2)

fig.update_xaxes(tickangle=-45, dtick=3, type='category', row=1, col=1)
fig.update_xaxes(title_text='Points Per Game', row=1, col=2)
fig.update_xaxes(title_text='Assists Per Game', row=2, col=1)
fig.update_xaxes(title_text='Age', row=2, col=2)
fig.update_yaxes(title_text='Per Game Avg', row=1, col=1)
fig.update_yaxes(title_text='Points Per Game', row=2, col=1)
fig.update_yaxes(title_text='Count', row=2, col=2)

fig.update_layout(template='plotly_dark', height=800, title_text='NBA Player Analytics Dashboard',
                  showlegend=True, legend=dict(orientation='h', y=1.02, x=0.5, xanchor='center'))
fig.show()

### 3.5 Deployed Dashboard — Interactivity

The notebook view above is hoverable but static. The deployed version uses Plotly.js in HTML with a season dropdown that re-renders all four charts:

```javascript
function update() {
    var season = document.getElementById('seasonSelect').value;
    // filter data by season, then:
    Plotly.newPlot('lineChart', lineTraces, lineLayout, config);
    Plotly.newPlot('barChart', [barTrace], barLayout, config);
    Plotly.newPlot('scatterChart', [scatterTrace], scatterLayout, config);
    Plotly.newPlot('histChart', [histTrace], histLayout, config);
}
```

CSV is loaded once at startup with [PapaParse](https://www.papaparse.com/) and held in memory.

Live at: [snehithn.com/dashboard.html](https://snehithn.com/dashboard.html)

### 3.6 Troubleshooting

Issues I ran into:

- **"unrecognized date" console errors**: Plotly tries to parse "2022-23" as a date. Fix: `xaxis: { type: 'category' }`
- **Bar chart blank on first load**: `Plotly.react()` can fail on uninitialized containers. Fix: use `Plotly.newPlot()` instead
- **CSV values as strings**: PapaParse defaults to strings. Fix: `dynamicTyping: true`
- **Charts not resizing**: Set `responsive: true` in the Plotly config

### 3.7 Deploy Instructions

Single HTML file, no backend needed.

**Repo structure:**
```
<your_folder>/
├── dashboard.html
└── data/
    └── nba_players.csv
```

**Data source**: [Kaggle — NBA Players Data](https://www.kaggle.com/datasets/justinas/nba-players-data)